# Google Subset — Chronological Splits and Baselines

This notebook prepares the leakage-safe train, validation, and test
artifacts used by downstream TraceAdvisor modeling.

## Responsibilities

1. Load the point-in-time execution feature artifact produced by Notebook 03.
2. Validate the canonical execution-level schema.
3. Create one deterministic chronological 70/15/15 split.
4. Keep the final test partition untouched during model and policy selection.
5. Generate Request, Mean, P90, P95, and P99 recommendation columns.
6. Apply the configured cold-start fallback policy.
7. Save stable train, validation, and test feature artifacts.

Historical profiles are created in Notebook 03 and are not recomputed here.

Evaluation formulas such as violation, bounded utilization, waste,
and shortfall are handled by the shared evaluation module.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

In [ ]:
# Locate repository root.
REPO_ROOT = Path.cwd()

while (
    REPO_ROOT.name != "masters-project"
    and REPO_ROOT.parent != REPO_ROOT
):
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate masters-project repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)

In [ ]:
from src.features.split_utils import chronological_split

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXECUTIONS_PATH = (
    DATA_DIR
    / "google_subset_executions_v1.parquet"
)

# TODO:
# Replace with Sofia's final Notebook 03 output filename
# once that contract is frozen.
PROFILE_FEATURES_PATH = (
    DATA_DIR
    / "google_subset_execution_features_v1.parquet"
)

TRAIN_FEATURES_PATH = (
    DATA_DIR
    / "train_features.parquet"
)

VAL_FEATURES_PATH = (
    DATA_DIR
    / "val_features.parquet"
)

TEST_FEATURES_PATH = (
    DATA_DIR
    / "test_features.parquet"
)

METADATA_PATH = (
    DATA_DIR
    / "metadata.json"
)

print("Data directory:", DATA_DIR)

In [ ]:
BASE_REQUIRED_COLUMNS = [
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

def validate_base_schema(
    df: pd.DataFrame,
) -> None:
    missing_columns = [
        column
        for column in BASE_REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required canonical columns: "
            f"{missing_columns}"
        )

    if df["execution_id"].isna().any():
        raise ValueError(
            "execution_id contains missing values."
        )

    if df["execution_id"].duplicated().any():
        raise ValueError(
            "execution_id must be unique."
        )

    if df["execution_start_time_us"].isna().any():
        raise ValueError(
            "execution_start_time_us contains missing values."
        )

    print(
        "Canonical base schema validation passed."
    )

## Notebook 03 Input Dependency

Notebook 04 consumes the execution-level point-in-time feature table
created by Notebook 03.

The final input should contain:

- `execution_id`
- `recurrence_key`
- `execution_start_time_us`
- current execution request fields
- current execution peak resource targets
- history depth
- past-only CPU history statistics
- past-only memory history statistics
- past-only runtime/stability features

Notebook 04 does not rebuild historical profiles.

In [ ]:
if PROFILE_FEATURES_PATH.exists():
    execution_features = pd.read_parquet(
        PROFILE_FEATURES_PATH
    )

    print(
        "Loaded:",
        execution_features.shape,
    )

    display(
        execution_features.head()
    )
else:
    print(
        "Notebook 03 artifact is not available yet:"
    )
    print(PROFILE_FEATURES_PATH)

In [ ]:
if PROFILE_FEATURES_PATH.exists():

    validate_base_schema(
        execution_features
    )

    (
        train_df,
        val_df,
        test_df,
        split_metadata,
    ) = chronological_split(
        execution_features,
        time_col="execution_start_time_us",
        id_col="execution_id",
        train_fraction=0.70,
        val_fraction=0.15,
    )

    print(
        "Train:",
        len(train_df),
    )

    print(
        "Validation:",
        len(val_df),
    )

    print(
        "Test:",
        len(test_df),
    )

In [ ]:
if PROFILE_FEATURES_PATH.exists():

    assert len(train_df) > 0
    assert len(val_df) > 0
    assert len(test_df) > 0

    assert train_df["execution_id"].is_unique
    assert val_df["execution_id"].is_unique
    assert test_df["execution_id"].is_unique

    assert set(train_df["execution_id"]).isdisjoint(
        set(val_df["execution_id"])
    )

    assert set(train_df["execution_id"]).isdisjoint(
        set(test_df["execution_id"])
    )

    assert set(val_df["execution_id"]).isdisjoint(
        set(test_df["execution_id"])
    )

    assert (
        train_df["execution_start_time_us"].max()
        <= val_df["execution_start_time_us"].min()
    )

    assert (
        val_df["execution_start_time_us"].max()
        <= test_df["execution_start_time_us"].min()
    )

    print("Split integrity checks passed.")

In [ ]:
if PROFILE_FEATURES_PATH.exists():

    split_summary = pd.DataFrame(
        {
            "split": [
                "train",
                "validation",
                "test",
            ],
            "rows": [
                len(train_df),
                len(val_df),
                len(test_df),
            ],
            "start_time_us": [
                train_df["execution_start_time_us"].min(),
                val_df["execution_start_time_us"].min(),
                test_df["execution_start_time_us"].min(),
            ],
            "end_time_us": [
                train_df["execution_start_time_us"].max(),
                val_df["execution_start_time_us"].max(),
                test_df["execution_start_time_us"].max(),
            ],
        }
    )

    display(split_summary)

In [ ]:
metadata = {
    "dataset": "google_subset_v1",
    "execution_id_field": "execution_id",
    "recurrence_key_field": "recurrence_key",
    "timestamp_field": "execution_start_time_us",
    "requested_cpu_field": "requested_cpu_ncu",
    "requested_memory_field": "requested_memory_normalized",
    "target_cpu_field": "peak_cpu_ncu",
    "target_memory_field": "peak_memory_normalized",
    "split": {
        "type": "chronological",
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
    },
    "history_rule": (
        "Only information available before each execution "
        "may be used in historical features."
    ),
    "test_policy": (
        "The final test split is not used for model selection, "
        "feature selection, min_runs selection, threshold selection, "
        "or hyperparameter tuning."
    ),
}

## Cold-Start Policy

Notebook 03 provides the amount of historical recurrence information
available before each execution.

Notebook 04 owns the policy decision for insufficient history.

`min_runs` will remain configurable and will be selected using training
and validation data only. The final test set will not be used to choose
`min_runs`.

In [ ]:
MIN_RUNS_CANDIDATES = [
    1,
    3,
    5,
    10,
]

print(
    "Candidate min_runs values:",
    MIN_RUNS_CANDIDATES,
)